In [1]:
import torch
import sys
print("Interpreter:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Interpreter: d:\maga25\VKRTimeSeries\.venv\Scripts\python.exe
CUDA available: True
GPU name: NVIDIA GeForce GTX 1660


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")
if device.type == 'cuda':
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")
    print(f"Всего видеопамяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Используемое устройство: cuda
Видеокарта: NVIDIA GeForce GTX 1660
Всего видеопамяти: 6.44 GB


In [3]:
# Standard
import random

import numpy as np
import pandas as pd
import torch

# Third Party
from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments,
)

# First Party
from tsfm_public.toolkit.dataset import ForecastDFDataset
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.util import select_by_index

d:\maga25\VKRTimeSeries\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

------------------------------------------------------------------------


Дообучение на датасете weather

In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import pandas as pd
import numpy as np
import torch
import random

torch.set_default_device('cpu')

from transformers import (
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.dataset import ForecastDFDataset

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

context_length = 168          
forecast_horizon = 96
patch_length = 12             # 168/12 = 14 патчей
batch_size = 8                
num_epochs = 25               
dataloader_workers = 0
pin_memory = False

df = pd.read_csv('data/weather.csv', parse_dates=['date'])
forecast_columns = [col for col in df.columns if col != 'date']

n = len(df)
train_end = int(0.7 * n)
valid_end = int(0.8 * n)

train_data = df.iloc[:train_end]
valid_data = df.iloc[train_end:valid_end]
test_data = df.iloc[valid_end:]

print(f"Train: {len(train_data)}, Valid: {len(valid_data)}, Test: {len(test_data)}")

tsp_weather = TimeSeriesPreprocessor(
    timestamp_column='date',
    id_columns=[],
    target_columns=forecast_columns,
    scaling=True,
)
tsp_weather.train(train_data)

train_dataset = ForecastDFDataset(
    tsp_weather.preprocess(train_data),
    id_columns=[],
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)
valid_dataset = ForecastDFDataset(
    tsp_weather.preprocess(valid_data),
    id_columns=[],
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)
test_dataset = ForecastDFDataset(
    tsp_weather.preprocess(test_data),
    id_columns=[],
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)

print(f"Train samples: {len(train_dataset)}, Valid: {len(valid_dataset)}, Test: {len(test_dataset)}")
if len(valid_dataset) == 0:
    raise ValueError("Валидационный датасет пуст.")

# адаптация
model_path = "./patchtst_etth1_model"
old_config = PatchTSTConfig.from_pretrained(model_path)

new_config = PatchTSTConfig(
    num_input_channels=len(forecast_columns),
    context_length=context_length,
    patch_length=patch_length,
    prediction_length=forecast_horizon,
    d_model=128,
    num_attention_heads=old_config.num_attention_heads,
    num_hidden_layers=old_config.num_hidden_layers,
    ffn_dim=old_config.ffn_dim,
    dropout=old_config.dropout,
    head_dropout=old_config.head_dropout,
    pooling_type=old_config.pooling_type,
    channel_attention=old_config.channel_attention,
    scaling=old_config.scaling,
    loss=old_config.loss,
    pre_norm=old_config.pre_norm,
    norm_type=old_config.norm_type,
    do_mask_input=True,
    mask_ratio=0.2,              
)

model = PatchTSTForPrediction.from_pretrained(
    model_path,
    config=new_config,
    ignore_mismatched_sizes=True
)
model = model.to('cpu')
print("✅ Модель адаптирована для Weather (21 канал) с маскировкой")


train_args = TrainingArguments(
    output_dir="./checkpoint/patchtst/finetune/weather_full/",
    learning_rate=1e-4,
    num_train_epochs=num_epochs,
    do_eval=True,
    eval_strategy="epoch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    dataloader_num_workers=dataloader_workers,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    weight_decay=0.01,
    label_names=["future_values"],
    fp16=False,
    dataloader_pin_memory=pin_memory,
    logging_steps=50,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=5,    
    early_stopping_threshold=0.001,
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    callbacks=[early_stopping],
)

print("\n Дообучение на полном датасете Weather (CPU, 25 эпох)...")
trainer.train()

print("\n Оценка на тесте:")
test_metrics = trainer.evaluate(test_dataset)
print(f"Test loss (MSE в нормированном пространстве): {test_metrics['eval_loss']:.4f}")

# Сохранение
model.save_pretrained("./patchtst_weather_finetuned_full")
tsp_weather.save_pretrained("./patchtst_weather_preprocessor_full")

Train: 36887, Valid: 5269, Test: 10540
Train samples: 36624, Valid: 5006, Test: 10277


[transformers] Setting `do_mask_input` parameter to False.
Loading weights: 100%|██████████| 71/71 [00:00<00:00, 1268.50it/s]
[transformers] PatchTSTForPrediction LOAD REPORT from: ./patchtst_etth1_model
Key                                           | Status   |                                                                                            
----------------------------------------------+----------+--------------------------------------------------------------------------------------------
model.encoder.positional_encoder.position_enc | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([42, 128]) vs model:torch.Size([157, 128])  
head.projection.weight                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([96, 5376]) vs model:torch.Size([96, 20096])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


✅ Модель адаптирована для Weather (21 канал) с маскировкой

 Дообучение на полном датасете Weather (CPU, 25 эпох)...


Epoch,Training Loss,Validation Loss
1,0.327867,0.412301
2,0.303602,0.414623
3,0.389455,0.498796
4,0.371230,0.470163
5,0.308692,0.457776
6,0.290073,0.467962


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 48.09it/s]



 Оценка на тесте:


Training Loss,Validation Loss,Epoch
0.290073,0.160152,6


Test loss (MSE в нормированном пространстве): 0.1602


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 31.81it/s]
INFO:p-8716:t-4512:processor.py:save_pretrained:Feature extractor saved in ./patchtst_weather_preprocessor_full\preprocessor_config.json


['./patchtst_weather_preprocessor_full\\preprocessor_config.json']